# The BERT rows, on a T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yougijain/text-vs-timing/blob/master/notebooks/bert_colab.ipynb)

Stage 5 of [`RUNBOOK.md`](https://github.com/yougijain/text-vs-timing/blob/master/RUNBOOK.md):
fine-tunes `bert-base-uncased`, with and without the timestamp, on the same corpus
the linear grid ran on, and regenerates `RESULTS.md` and the published page with
all six rows.

**Run the linear grid first** (runbook stages 1–4, on a laptop). You want to know
what the cheap models say before spending a GPU hour on whether the expensive one
beats them.

**You need:** the `datasets/posts.csv` from that run (it is gitignored, so it is not
in the clone) and a T4 runtime. This notebook asks for one; check
*Runtime → Change runtime type* says **T4 GPU**.

About 30–60 minutes. Keep the tab open: free Colab disconnects idle sessions.

## 1. Check the GPU

If this errors or does not say `Tesla T4`, change the runtime type and run it again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"

## 2. Clone and install

Colab already has a CUDA build of torch, so this is quick. Safe to re-run.

In [ ]:
%cd /content
![ -d text-vs-timing ] || git clone -q https://github.com/yougijain/text-vs-timing.git
%cd /content/text-vs-timing
!git pull -q && git log --oneline -1
!pip install -q -r requirements.txt

## 3. Upload the corpus

Pick `datasets/posts.csv` from your laptop: the *same* file the linear grid ran
on. Re-fetching would burn API quota and return different posts. The provenance
sidecar came down with the clone, so it still matches.

In [ ]:
import os
import shutil
from google.colab import files

os.makedirs("datasets", exist_ok=True)
uploaded = files.upload()
shutil.move(next(iter(uploaded)), "datasets/posts.csv")
!python -m data.inspect_dataset --dataset datasets/posts.csv | head -3

## 4. The run

The whole grid, so the regenerated report has all six rows. Defaults are already
what you want: `bert-base-uncased`, temporal split, median labels, AMP on.

If the loss goes to `nan`, add `--no-amp` and run it again.

In [ ]:
!python run_experiment.py --dataset datasets/posts.csv \
    --epochs 3 --max-length 256 --batch-size 16 \
    --embeddings --site 2>&1 | tee run.log

## 5. Record the training time, then download

`Training finished in Ns` is printed and stored nowhere else: it reaches neither
`benchmark.json` nor `RESULTS.md`. It is the total across all epochs, once per
BERT row. Write it down; it goes in the commit message.

The zip leaves out the checkpoints (about 440 MB each, and anyone can retrain them).

In [ ]:
!grep "Training finished" run.log
!zip -qr bert_run.zip RESULTS.md docs outputs run.log -x "outputs/checkpoints/*"
files.download("bert_run.zip")

## 6. Back on the laptop

Unzip `bert_run.zip` over the checkout, then read `RESULTS.md` again, whole
(runbook stage 6), before committing it:

```bash
git add RESULTS.md docs/
git commit -m "results: BERT rows on a Colab T4"
git push origin master
```

Put the GPU and the training times in the commit message, and change the README
status row that says the BERT rows have not been run.